# 第 1 周末练习 —— 技术问答解释器（OpenRouter + 本地 Ollama）

## 练习目标（理念）

构建一个小工具：输入技术问题，分别用云端模型与本地 Llama **流式**给出解释。  
这是你在课程期间自己也能天天用的工具。

- **输入**：一段技术问题（本例问 `yield from` + 集合推导）
- **输出**：Markdown 解释，边生成边用 `update_display` 刷新
- **两条后端**：OpenRouter 上的 `gpt-4o-mini`，以及本机 Ollama 的 `llama3.2`（OpenAI 兼容 `/v1`）

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions + streaming | `stream=True` + 逐 chunk 更新显示 |
| system / user messages | `SYSTEM_PROMPT` + `question` |
| 云端 API | OpenRouter `base_url` + `OPENAI_API_KEY` |
| 本地 Ollama | `http://localhost:11434/v1` |

## 怎么跑

1. `.env` 里配置可用于 OpenRouter 的 `OPENAI_API_KEY`（本练习校验前缀 `sk-or-v1`）
2. 若跑 Llama：本机 Ollama 已启动并 pull `llama3.2`
3. 从上到下运行；在「提问」格改写 `question` 后再跑 GPT / Llama 两格


In [ ]:
# ========== 导入：环境变量、展示、OpenAI 兼容客户端 ==========

# 标准库 os：读环境变量里的 API Key
import os
# 标准库 json：本格导入备用（后面流式路径未强制用到）
import json
# load_dotenv：把 .env 读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# Markdown / display / update_display：笔记本里流式刷新 Markdown 输出
from IPython.display import Markdown, display, update_display
# OpenAI 客户端：既可打 OpenRouter，也可打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI


In [4]:
# ========== 常量：模型名字集中管理 ==========

# OpenRouter / 云端小模型名
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名（须已 pull，且与服务端登记名一致）
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境：加载 .env，校验 Key，创建指向 OpenRouter 的客户端 ==========

# override=True：.env 里的值覆盖已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OPENAI_API_KEY（此处实际给 OpenRouter 用）
api_key = os.getenv('OPENAI_API_KEY')

# 粗测：OpenRouter 密钥常以 sk-or-v1 开头且长度 > 10；提示文案保持英文
if api_key and api_key.startswith('sk-or-v1') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# base_url 指到 OpenRouter 的 OpenAI 兼容端点；密钥仍走 api_key 参数
openai = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)


API key looks good so far


In [13]:
# ========== 提问 + system prompt：改 question 就能问新问题 ==========

# 发给模型的问题保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

# system：定助手风格；英文指令保留
SYSTEM_PROMPT = "You are a helpful coding assistant. Explain technical concepts clearly and concisely. Include code examples. Use markdown for formatting."


In [14]:
# ========== 云端 GPT：流式 Chat Completions，边收边刷新 Markdown ==========

def get_authors():
    # stream=True：返回可迭代的增量事件，而不是等整段生成完
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    # response：累积已收到的全文，用于反复 update_display
    response = ""
    # 先占位一个空 Markdown，拿到 display_id 以便原地更新
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # delta.content 可能为 None（例如角色/结束片），用 or '' 避免拼 None
        response += chunk.choices[0].delta.content or ''
        # 用同一 display_id 刷新，实现「打字机」效果
        update_display(Markdown(response), display_id=display_handle.display_id)

# 立刻调用：运行本格即向 OpenRouter 提问
get_authors()


The code snippet you provided uses the `yield from` statement in Python, combined with a set comprehension. Here’s a breakdown of what it does and why:

### Breakdown of the Code

1. **Set Comprehension**: 
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   - This part of the code creates a set of authors from a list (or any iterable) called `books`.
   - It iterates over each `book` in `books`.
   - For each `book`, it tries to get the value associated with the key `"author"` using `book.get("author")`. 
   - The `if book.get("author")` condition ensures that only books that actually have an author (`author` is not `None` or an empty string) are considered.
   - The result is a set containing unique authors from the `books` iterable.

2. **`yield from`**: 
   ```python
   yield from
   ```
   - `yield from` is used in generator functions to yield all values from another iterable. This allows a generator to delegate part of its operations to another generator or iterable.
   - In this case, it yields each author from the set of authors generated by the set comprehension.

### Why Use This Code?

- **Eliminates Duplicates**: By using a set comprehension, the code automatically filters out duplicate authors. This is useful when you want a unique collection of authors without manually checking for duplicates.
- **Efficient Iteration**: The `yield from` expression allows the function to yield each author sequentially, which can be memory efficient for large datasets since it does not require storing all authors in a list beforehand.
- **Readable and Concise**: This approach is clearer and more concise than using explicit loops to filter and yield authors.

### Example

Here's an example to illustrate how this might be used in practice:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Example data
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": "Author 1"},  # Duplicate author
    {"title": "Book D"},  # No author
    {"title": "Book E", "author": "Author 3"},
]

# Using the generator
for author in get_unique_authors(books):
    print(author)
```

### Output

```
Author 1
Author 2
Author 3
```

In this example, the generator yields each unique author found in the `books` collection, demonstrating the functionality of the provided code.

In [15]:
# ========== 本地 Llama：OpenAI 兼容客户端指向 Ollama /v1，同样流式展示 ==========

# api_key 对本地 Ollama 通常任意非空即可；base_url 指本机 11434 的 v1 兼容层
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

def get_authors_with_llama():
    # 与 GPT 路径相同：stream=True + update_display
    stream = ollama_client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

# 运行本格：向本地 llama3.2 提问并流式显示
get_authors_with_llama()


**Code Breakdown**
===============

The given code snippet uses the `yield from` syntax to generate a sequence of values.

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Here's what it does:

*   **Dictionary comprehension**: `{book.get("author") for book in books if book.get("author")}` is a dictionary comprehension. It creates a new dictionary containing only the author names.
    *   `for book in books`: Iterates over each item (`book`) in the list or iterable `books`.
    *   `if book.get("author")`: Filters the items to include only those that have an "author" key with a value of some type (e.g., string).
    *   `book.get("author")`: Retrieves the author name from each valid item.
*   **`yield from`**: This syntax is used to delegate iteration to another iterable.
    *   It takes any iterable (like a dictionary comprehension) and yields its values directly, without having to store them in memory.

**Why `yield from`?**
-------------------

You might wonder why use `yield from` instead of simply iterating over the dictionary comprehension. The main advantage is that `yield from` can significantly reduce memory usage for large datasets or infinite sequences.

By using `yield from`, we only yield one value at a time, allowing us to generate the sequence without storing all values in memory. This makes the code more memory-efficient and suitable for large inputs.

**Example Usage**
----------------

```python
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2", "author": "Jane Smith"},
    {"title": "Book 3"}
]

for author in yield from {book["author"] for book in books if "author" in book}:
    print(author)
```

This will output:

```
John Doe
Jane Smith
```